In [ ]:
%pip install rasterio pillow

In [ ]:
# ======================================================================
# STANDALONE JOB: generate missing orthomosaic PNG previews.
#
# 1. Reads the ortho table and keeps only rows whose orthomosaic exists
#    (ortho_exists == True).
# 2. Takes the .tif path directly from the 'ortho_file_path' column.
# 3. Checks whether the sibling .png already exists; generates it only
#    when missing, storing it next to the .tif.
#
# Requires rasterio and Pillow on the cluster:
#   %pip install rasterio pillow
# ======================================================================
import os, shutil
import numpy as np
import rasterio
from rasterio.enums import Resampling, ColorInterp
from PIL import Image
import pyspark.sql.functions as F

# =============================== PARAMETERS ============================
CATALOG = "apse2_prod_irri_fg_catalog_7474658213144266"  # change
SCHEMA  = "tier1_raw"                                     # change

ortho_table = f"{CATALOG}.{SCHEMA}.drone_ortho_table"

ORTHO_PATH_COLUMN = "ortho_file_path"   # column holding the .tif path
FORCE_REGENERATE  = False               # True to rebuild PNGs even if they exist

# ------------------------------ PNG config ----------------------------
MAX_DIM     = 4096        # max PNG size (px). Set to None for full resolution.
STRETCH_PCT = (2, 98)     # contrast stretch by percentiles
# For multispectral: which spectral bands map to R, G, B (1-based).
# DJI P4M / MicaSense order is usually Blue, Green, Red, RedEdge, NIR,
# so natural color = (3, 2, 1). Change this if your camera differs.
MS_RGB_BANDS = (3, 2, 1)
# ======================================================================


# --------------------------- PNG generation ---------------------------
def get_spectral_bands(src):
    """Return (spectral_bands_1based, alpha_index_or_None)."""
    spectral, alpha = [], None
    for i, ci in enumerate(src.colorinterp, start=1):
        if ci == ColorInterp.alpha:
            alpha = i
        else:
            spectral.append(i)
    return spectral, alpha


def stretch_band(band, low, high):
    """Rescale a single band to 8-bit using the given low/high bounds."""
    if high <= low:
        return np.zeros(band.shape, dtype="uint8")
    out = np.clip((band.astype("float32") - low) / (high - low), 0, 1) * 255.0
    return out.astype("uint8")


def tif_to_png(src_tif, dst_png, sensor_type):
    with rasterio.open(src_tif) as src:
        # Downscale factor so the longest side fits within MAX_DIM.
        scale = min(1.0, MAX_DIM / max(src.width, src.height)) if MAX_DIM else 1.0
        out_h = max(1, int(round(src.height * scale)))
        out_w = max(1, int(round(src.width * scale)))

        spectral, alpha_idx = get_spectral_bands(src)

        # Pick the R, G, B bands.
        if sensor_type == "MS" and len(spectral) >= 3:
            chosen = [spectral[b - 1] for b in MS_RGB_BANDS]
        elif len(spectral) >= 3:
            chosen = spectral[:3]
        else:                       # single band -> replicate into grayscale
            chosen = [spectral[0]] * 3

        data = src.read(chosen, out_shape=(3, out_h, out_w),
                        resampling=Resampling.bilinear).astype("float32")

        # Mask of valid pixels (from alpha, nodata or NaN).
        if alpha_idx is not None:
            valid = src.read(alpha_idx, out_shape=(out_h, out_w),
                             resampling=Resampling.nearest) > 0
        elif src.nodata is not None:
            valid = np.all(data != src.nodata, axis=0)
        else:
            valid = np.ones((out_h, out_w), dtype=bool)
        valid &= np.all(np.isfinite(data), axis=0)

        # Per-band contrast stretch using only valid pixels.
        rgb = np.zeros((out_h, out_w, 3), dtype="uint8")
        for k in range(3):
            vals = data[k][valid]
            if vals.size:
                low, high = np.percentile(vals, STRETCH_PCT)
                rgb[..., k] = stretch_band(data[k], low, high)

        # RGBA output: empty/nodata areas become transparent.
        alpha_ch = np.where(valid, 255, 0).astype("uint8")
        Image.fromarray(np.dstack([rgb, alpha_ch]), mode="RGBA").save(dst_png)


def normalize_path(p):
    """Make a stored path usable by os/shutil (dbfs:/ -> /dbfs/)."""
    return p.replace("dbfs:/", "/dbfs/") if p else p


def detect_sensor_type(tif_path):
    """Infer MS vs RGB from the filename (MS.tif -> MS, otherwise RGB)."""
    return "MS" if os.path.basename(tif_path).upper().startswith("MS") else "RGB"


# --------------------- Select orthos that exist -----------------------
raw_ortho_df = spark.table(ortho_table)

# Same rule as your inventory: an ortho is "finished" when ortho_exists is True.
if "ortho_exists" in raw_ortho_df.columns:
    finish_df = raw_ortho_df.filter(F.col("ortho_exists") == True)
else:
    finish_df = raw_ortho_df

# Pull the .tif paths directly from the ortho table.
ortho_rows = (finish_df
              .select(ORTHO_PATH_COLUMN)
              .where(F.col(ORTHO_PATH_COLUMN).isNotNull())
              .distinct()
              .collect())

ortho_paths = [r[ORTHO_PATH_COLUMN] for r in ortho_rows]

print("-" * 40)
print(f"Orthomosaics to inspect: {len(ortho_paths)}")
print("-" * 40)


# --------------------------- Main loop --------------------------------
generated, skipped, errors = 0, 0, 0

for raw_path in ortho_paths:
    tif = normalize_path(raw_path)

    if not tif.lower().endswith((".tif", ".tiff")):
        print(f"--   Skipping non-tif path: {tif}")
        continue
    if not os.path.exists(tif):
        print(f"--   File not found on disk: {tif}")
        continue

    sensor_type = detect_sensor_type(tif)
    png_path    = os.path.splitext(tif)[0] + ".png"   # stored next to the .tif

    # THE CHECK: only generate when the PNG is missing (unless forced).
    if os.path.exists(png_path) and not FORCE_REGENERATE:
        skipped += 1
        print(f"SKIP {png_path}  (already exists)")
        continue

    # Work on local disk to avoid Volume I/O errors.
    tmp_tif = "/tmp/_ortho_src.tif"
    tmp_png = "/tmp/_ortho_out.png"
    try:
        shutil.copy2(tif, tmp_tif)
        tif_to_png(tmp_tif, tmp_png, sensor_type)
        shutil.copy2(tmp_png, png_path)
        generated += 1
        print(f"OK   {tif}  ->  {png_path}")
    except Exception as e:
        errors += 1
        print(f"ERR  {tif}: {e}")
    finally:
        for f in (tmp_tif, tmp_png):
            if os.path.exists(f):
                os.remove(f)

# ------------------------------ Summary -------------------------------
summary = (f"Done. PNG generated: {generated} | "
           f"Skipped (already had PNG): {skipped} | Errors: {errors}")
print("\n" + summary)
dbutils.notebook.exit(summary)